# 🧩 Nodes — Arguments, Config & Runtime Context

## Learning Objectives
In this notebook, you will learn:
1. **Plain nodes** — functions that only receive state
2. **Config-aware nodes** — accessing `RunnableConfig` for thread IDs and metadata
3. **Runtime context nodes** — injecting external dependencies via `Runtime[Schema]`
4. **Context schemas** — type-safe runtime context with `context_schema`

## Prerequisites
- `langgraph`, `langchain-core` installed
- Understanding of `StateGraph` basics (notebooks `02`–`06`)

---
## 🔧 Part 1: Environment Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from typing import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime

print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


---
## 📋 Part 2: Define State & Context Schemas

We define two schemas:
- **`GraphState`** — the graph's shared mutable state
- **`ContextSchema`** — runtime-only data (not stored in state)

In [2]:
# ============================================================================
# SCHEMAS: Graph state and runtime context
# ============================================================================
class GraphState(TypedDict):
    """Represents the state of our graph."""
    input: str
    results: str


class ContextSchema(TypedDict):
    """Contains information available at runtime, not stored in the state."""
    user_id: str

print("✅ Schemas defined!")

✅ Schemas defined!


---
## ⚙️ Part 3: Define the Nodes

Three node types demonstrating different argument patterns:

### 3.1 Plain Node — `state` only
The simplest form: receives the current state, returns a partial update.

In [3]:
# ============================================================================
# NODE A: Plain node (state only)
# ============================================================================
def plain_node(state: GraphState) -> dict:
    """A node that simply updates the state's 'results' key."""
    print("Executing plain_node...")
    return {"results": f"Hello, {state['input']}!"}

### 3.2 Config Node — `state` + `RunnableConfig`

Access metadata like `thread_id` from the `RunnableConfig` passed at invocation time.

In [4]:
# ============================================================================
# NODE B: Config-aware node
# ============================================================================
def node_with_config(state: GraphState, config: RunnableConfig) -> dict:
    """Accesses a value from the RunnableConfig."""
    print("Executing node_with_config...")
    thread_id = config.get("configurable", {}).get("thread_id")
    print(f"-> Accessed 'thread_id' from config: {thread_id}")
    return {"results": "Config access successful."}

### 3.3 Runtime Node — `state` + `Runtime[ContextSchema]`

Inject external dependencies (DB connections, user info) via a typed runtime
context object passed at invocation time.

In [5]:
# ============================================================================
# NODE C: Runtime context node
# ============================================================================
def node_with_runtime(state: GraphState, runtime: Runtime[ContextSchema]) -> dict:
    """Accesses a value from the custom runtime context."""
    print("Executing node_with_runtime...")
    user_id = runtime.context["user_id"]
    print(f"-> Accessed 'user_id' from runtime: {user_id}")
    return {"results": "Runtime access successful."}

print("✅ All nodes defined!")

✅ All nodes defined!


---
## 🔗 Part 4: Build & Run the Graph

We pass `context_schema=ContextSchema` to `StateGraph` so it knows the shape
of the runtime context. At invocation time, we provide both a `config` dict
and a `context` dict.

In [6]:
# ============================================================================
# GRAPH CONSTRUCTION: Build with context schema
# ============================================================================
builder = StateGraph(GraphState, context_schema=ContextSchema)

builder.add_node("plain_node", plain_node)
builder.add_node("config_node", node_with_config)
builder.add_node("runtime_node", node_with_runtime)

builder.add_edge(START, "plain_node")
builder.add_edge("plain_node", "config_node")
builder.add_edge("config_node", "runtime_node")
builder.add_edge("runtime_node", END)

app = builder.compile()

print("✅ Graph compiled!")

✅ Graph compiled!


In [7]:
# ============================================================================
# EXECUTION: Invoke with state, config, and context
# ============================================================================
initial_state = {"input": "World"}

run_config = {
    "configurable": {
        "thread_id": "user-1234",
    }
}

print("--- Invoking the graph ---")
final_state = app.invoke(
    input=initial_state,
    config=run_config,
    context={"user_id": "alice_smith"}
)

print("\n--- Final State of the Graph ---")
print(final_state)

--- Invoking the graph ---
Executing plain_node...
Executing node_with_config...
-> Accessed 'thread_id' from config: user-1234
Executing node_with_runtime...
-> Accessed 'user_id' from runtime: alice_smith

--- Final State of the Graph ---
{'input': 'World', 'results': 'Runtime access successful.'}


---
## 📝 Summary

In this notebook, we learned:

### 1. Three Node Argument Patterns
| Pattern | Signature | Use Case |
|---------|-----------|----------|
| Plain | `f(state)` | Simple state transformations |
| Config | `f(state, config)` | Access thread IDs, metadata |
| Runtime | `f(state, runtime)` | Inject DB connections, user context |

### 2. Context Schema
- Defined as a `TypedDict` and passed to `StateGraph(context_schema=...)`
- Provided at `invoke(context={...})` time

### Next Steps
- Learn **Edges** and conditional routing (notebook `08-edges`)
- Explore **Runtime Context** in depth (notebook `09-runtime-context`)